# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ShlokNoval/Flyrank-Internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

**Unit of analysis:** One row in the feature frame represents one pseudonymized content item (page) for a specific client.

**Time window:** Features will be drawn from the mid-panel month of March 2026 (`month=2026-03`). The target outcome (the decline we want to predict) will be measured in the following 30-day window (April 2026).

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Fields: feature / label / context / excluded

- **Feature:** `imp_mar` (impressions in March), `clk_mar` (clicks in March), `pos_mar` (avg position in March) — all observed and finalized before the decision point at the end of March.
- **Label / proxy:** `is_declining_april` — the outcome we are predicting, calculated from April data. Never used as a feature.
- **Context:** `content_hash_id`, `client_hash_id` — used strictly for joining and grouped splitting, never for the model to learn from.
- **Excluded:** `ga4_sessions` when `ga4_data_available` is FALSE — excluded because zero-filled rows before analytics tracking started are not real zero engagement.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [ ]:
import duckdb
import os
import getpass

HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass('Paste your Hugging Face READ token (hf_...): ')

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")
REL = 'hf://datasets/FlyRank/internship-warehouse'

# 1. Grain Verification: Check that one row = one content item per client per day
print("1. Grain Verification:")
grain_check = con.sql(f"""
    SELECT client_hash_id, content_hash_id, report_date, COUNT(*) as c 
    FROM read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet') 
    GROUP BY report_date, client_hash_id, content_hash_id 
    HAVING c > 1 LIMIT 5
""").df()
print(f"Rows violating grain (should be 0): {len(grain_check)}\n")

# 2. Row count and date span for month=2026-03
print("2. Row count and date span:")
span_check = con.sql(f"""
    SELECT COUNT(*) as total_rows, MIN(report_date) as min_date, MAX(report_date) as max_date 
    FROM read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')
""").df()
print(span_check, "\n")

# 3. Availability of GA4 data
print("3. GA4 Data Availability:")
avail_check = con.sql(f"""
    SELECT 
        COUNT(*) as total_rows, 
        SUM(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END) as ga4_available_rows 
    FROM read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')
""").df()
print(avail_check, "\n")

# Feature Frame & The Trap
print("4. Features and Leakage Trap:")
features_and_trap = con.sql(f"""
    SELECT 
        client_hash_id, content_hash_id,
        SUM(gsc_impressions) as imp_mar, -- Feature 1: knowable at end of March
        SUM(gsc_clicks) as clk_mar, -- Feature 2: knowable at end of March
        AVG(gsc_avg_position) as pos_mar, -- Feature 3: knowable at end of March
        SUM(CASE WHEN dayofweek(report_date) IN (0, 6) THEN gsc_impressions ELSE 0 END) as weekend_imp, -- Feature 4: knowable at end of March
        SUM(CASE WHEN ga4_data_available IS TRUE THEN ga4_sessions ELSE 0 END) as sessions_mar -- Feature 5: knowable at end of March
        -- TRAP: If we added 'imp_april' here, the model would memorize the future. This is label leakage.
    FROM read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')
    GROUP BY client_hash_id, content_hash_id
    LIMIT 5
""").df()
print(features_and_trap)
print("\nTrap sprung and removed: Including data from the target window (April) causes a perfect but worthless score (leakage).")


## 4. Data limits

**Limitation:** The panel is highly unbalanced. Because `gsc_data_start` and `ga4_data_start` vary significantly by client, picking a fixed historical window like March 2026 means we may accidentally include clients who hadn't installed analytics yet. We must filter by the client's `ga4_data_start` rather than assuming zero traffic means poor performance.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.